### load

In [1]:
import numpy as np
import polars as pl
from polars import selectors as cs
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler,FunctionTransformer
from sklearn.compose import ColumnTransformer, make_column_selector, TransformedTargetRegressor
from sklearn.model_selection import train_test_split,cross_validate,KFold
from sklearn.metrics import root_mean_squared_log_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
%matplotlib inline

from add_features import add_modified_features


In [2]:
train = pl.read_csv('../data/train.csv',infer_schema_length=None, null_values='NA')
test = pl.read_csv('../data/test.csv',infer_schema_length=None, null_values='NA')

print(train.shape, test.shape)

(1460, 81) (1459, 80)


In [3]:
X = train.select(cs.numeric() - cs.by_name('Id','SalePrice'))
y = train['SalePrice']

X_test = test.select(cs.numeric() - cs.by_name('Id'))
X.glimpse()

Rows: 1460
Columns: 36
$ MSSubClass    <i64> 60, 20, 60, 70, 60, 50, 20, 60, 50, 190
$ LotFrontage   <i64> 65, 80, 68, 60, 84, 85, 75, null, 51, 50
$ LotArea       <i64> 8450, 9600, 11250, 9550, 14260, 14115, 10084, 10382, 6120, 7420
$ OverallQual   <i64> 7, 6, 7, 7, 8, 5, 8, 7, 7, 5
$ OverallCond   <i64> 5, 8, 5, 5, 5, 5, 5, 6, 5, 6
$ YearBuilt     <i64> 2003, 1976, 2001, 1915, 2000, 1993, 2004, 1973, 1931, 1939
$ YearRemodAdd  <i64> 2003, 1976, 2002, 1970, 2000, 1995, 2005, 1973, 1950, 1950
$ MasVnrArea    <i64> 196, 0, 162, 0, 350, 0, 186, 240, 0, 0
$ BsmtFinSF1    <i64> 706, 978, 486, 216, 655, 732, 1369, 859, 0, 851
$ BsmtFinSF2    <i64> 0, 0, 0, 0, 0, 0, 0, 32, 0, 0
$ BsmtUnfSF     <i64> 150, 284, 434, 540, 490, 64, 317, 216, 952, 140
$ TotalBsmtSF   <i64> 856, 1262, 920, 756, 1145, 796, 1686, 1107, 952, 991
$ 1stFlrSF      <i64> 856, 1262, 920, 961, 1145, 796, 1694, 1107, 1022, 1077
$ 2ndFlrSF      <i64> 854, 0, 866, 756, 1053, 566, 0, 983, 752, 0
$ LowQualFinSF  <i64> 0, 0, 0, 

### explore

In [ ]:
train.describe()

In [ ]:
nulls = train.select(cs.numeric()).to_pandas().isna().sum()
nulls[nulls > 0]

### col-prep mapping

In [ ]:
map_label = {
    'Street': {'Pave':1,'Grvl':0},
    'CentralAir': {'Y':1, 'N':0},
}

eq_val_label = {
    'Heating': 'GasA',
    'PavedDrive': 'Y',
}

is_na_label = [
    'Alley',
    'PoolQC',
    'Fence',
    'MiscFeature',
]

is_zero_label = [
    'LowQualFinSF',
    'BsmtFinSF2',
    'PoolArea',
    'BsmtHalfBath',
    'ScreenPorch',
    '3SsnPorch',
    'MiscVal',
]

ordinal_encoded = {
    'Condition1',
    'KitchenQual',
    'Functional',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
}

one_hot_encoded = [  # 最頻値で補完
    'MSZoning',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'ExterQual',
    'ExterCond',
    'Foundation',
    'HeatingQC',
    'SaleType',
    'SaleCondition',
    'Electrical',
    'MasVnrType',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'GarageType',
    'GarageFinish',
]

target_encoded = [
    'MSSubClass',
    'Neighborhood',
]

log_standardized = [ # 中央値で補完
    'LotFrontage',
    'LotArea',
    'YearBuilt',
    'GrLivArea',
    'BsmtUnfSF',
    'TotalBsmtSF',
    'BsmtFinSF1',
    'MasVnrArea',
    'EnclosedPorch',
    'HalfBath',
    'OpenPorchSF',
    'WoodDeckSF',
]

standardized = [ # 中央値で補完
    'OverallQual',
    'OverallCond',
    'BsmtFullBath',
    'FullBath',
    'BedroomAbvGr',
    'KitchenAbvGr',
    'TotRmsAbvGrd',
    'Fireplaces',
    'GarageYrBlt',
    'GarageCars',
    'GarageArea',
    'MoSold',
    'YrSold',
]

### pipeline

In [ ]:
# Target-Pipeline
log_standardize_y = Pipeline([
    ("log", FunctionTransformer(np.log1p, inverse_func=np.expm1, check_inverse=False)),
    ("scale", StandardScaler())
])

# Features Pipeline
log_standardize = Pipeline([
    ("log", FunctionTransformer(np.log1p, inverse_func=np.expm1, check_inverse=False)),
    ("scale", StandardScaler())
])

### add modified feature

In [ ]:
X_train, X_va, y_train, y_va = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X = add_modified_features(X)
X_test = add_modified_features(X_test)

### cross validation

In [ ]:
# base_model = Pipeline([
#     ("log_standard", log_standardize),
#     ("rfr", RandomForestRegressor(
#         n_estimators=200,
#         max_depth=5,
#         random_state=42,
#         n_jobs=-1))
# ])
base_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log_standard", log_standardize),
    ("ridge", Ridge(alpha=10))
])
model = TransformedTargetRegressor(
    regressor=base_model,
    transformer=log_standardize_y,
    check_inverse=False
)

folds = 5
cv = KFold(n_splits=folds, shuffle=True, random_state=42)
result = cross_validate(
    model, X_train, y_train,
    cv=cv,
    scoring="neg_root_mean_squared_log_error",
    return_train_score=True,
)

# スコア表示
train_scores = -result["train_score"]
valid_scores = -result["test_score"]

for i, (tr, va) in enumerate(zip(train_scores, valid_scores), 1):
    print(f"Fold {i}: train RMSLE = {tr:.4f} / valid RMSLE = {va:.4f}")
print(f"CV平均: train {train_scores.mean():.4f} / valid {valid_scores.mean():.4f}")

model.fit(X_train, y_train)
test_score = root_mean_squared_log_error(y_va, model.predict(X_va))
print(f"最終スコア(test RMSLE): {test_score:.4f}")

# スコアグラフ
df = pl.DataFrame({
    "Fold": [str(i) for i in range(folds)] * 2 + ["Final"],
    "RMSLE": list(train_scores) + list(valid_scores) + [test_score],
    "type": ["train"] * folds + ["valid"] * folds + ["test"],
})

plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="Fold", y="RMSLE", hue="type")
plt.title("RMSLE per Fold + Final Test Score")
plt.show()


### submit

In [ ]:
submit_df = pl.DataFrame({
    "Id": test['Id'],
    "SalePrice": model.predict(X_test)
})

In [ ]:
submit_df.write_csv("../submit.csv")